**Overview**  
This notebook describes EACN prediction with SVR model model. Firs, we perform its training on the whole dataset and save trained instance of the estimator. In the second part of the notebook you can calculate EACN of any compound you want - just follow prompt after "here the calculator starts"!

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

from rdkit import Chem
from rdkit.Chem import AllChem

import datamol as dm
from molfeat_padel.calc import PadelDescriptors


from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
from sklearn.pipeline import make_pipeline

First, let's get all features

In [5]:
with open('X_padel_dropped.pickle', 'rb') as inp:
    X_padel_dropped = pickle.load(inp)

In [6]:
with open('EACN.pickle', 'rb') as inp:
    target = pickle.load(inp)


Then we get pipeline with hyperparameters, found on the second step and train it on our data

In [11]:
pipe = make_pipeline(StandardScaler(), SVR(C = 7.008474598564191,
                                           coef0 = 0.5325084098836236,
                                           degree = 2,
                                           gamma = 'scale',
                                           kernel = 'poly'))

In [13]:
loo = LeaveOneOut()
predictions = cross_val_predict(estimator=pipe, X = X_padel_dropped, y = target, cv = loo)

In [17]:
print('R2 for LOO cross-validation is {}'.format(r2_score(predictions, target)))
print('RMSE for LOO cross-validation is {}'.format(root_mean_squared_error(predictions, target)))
print('MAE for LOO cross-validation is {}'.format(mean_absolute_error(predictions, target)))

R2 for LOO cross-validation is 0.8819657483435726
RMSE for LOO cross-validation is 2.2731508692006
MAE for LOO cross-validation is 1.3043313029772472


In [18]:
pipe.fit(X_padel_dropped, target)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('svr',
                 SVR(C=7.008474598564191, coef0=0.5325084098836236, degree=2,
                     kernel='poly'))])

In [19]:
with open('trained_pipe.pickle', 'wb') as out:
    pickle.dump(pipe, out)

Here calculator starts!

In [3]:
with open('trained_pipe.pickle', 'rb') as inp:
    pipe = pickle.load(inp)

In [4]:
with open('imputer.pickle', 'rb') as inp:
    imputer = pickle.load(inp)

In [5]:
smi = input('Please, enter SMILES of your molecule')
desc_calc = PadelDescriptors()
with dm.without_rdkit_log():
    descs = pd.DataFrame(desc_calc(smi), index = desc_calc.columns).transpose()

descs = pd.DataFrame(imputer.transform(descs), columns = descs.columns)
    
with open('features_to_drop_Padel.pickle', 'rb') as inp:
    features_to_drop = pickle.load(inp)

descs = descs.drop(columns = features_to_drop)
print('EACN of the molecule is {}'.format(pipe.predict(descs)))

Please, enter SMILES of your molecule C(Cl)(Cl)(Cl)


Failed to find the pandas get_adjustment() function to patch
Failed to patch pandas - PandasTools will have limited functionality


EACN of the molecule is [-14.01064743]
